<a href="https://colab.research.google.com/github/nosadchiy/public/blob/main/Obermeyer_beta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sport Obermeyer: patched search for MOQ-floor styles

This notebook patches the earlier logic.

## Objective
Choose round-1 quantities to:

1. meet the minimum order quantity (`MOQ`) for each style ordered in round 1,
2. meet the total first-round quantity requirement (`TOTAL_Q1`),
3. **equalize** the risk  
   \[
   P(D_i < q_{1i}+\text{MOQ})
   \]
   across **active** styles, and
4. when one or more included styles sit at the MOQ floor, choose the feasible partition that **minimizes the maximum included risk**, then minimizes the dispersion of included risks.

## States
Each style can be in one of three states:

- **deferred**: \(q_{1i}=0\)
- **bound**: \(q_{1i}=\text{MOQ}\)
- **active**: \(q_{1i}>\text{MOQ}\)

For active styles, equalization implies
\[
q_{1i}+\text{MOQ} = \mu_i + z\sigma_i
\quad\Rightarrow\quad
q_{1i} = \mu_i + z\sigma_i - \text{MOQ}.
\]

If \(A\) is the set of active styles and \(B\) is the set of bound styles, then with the total first-round quantity binding at equality,
\[
z =
\frac{\text{TOTAL\_Q1} - |B|\cdot \text{MOQ} - \sum_{i\in A}\mu_i + |A|\cdot \text{MOQ}}
     {\sum_{i\in A}\sigma_i}.
\]

Included-style risks are then:

- active style \(i\): \(\Phi(z)\)
- bound style \(i\):  
  \[
  P(D_i < 2\cdot \text{MOQ}) = \Phi\!\left(\frac{2\cdot\text{MOQ}-\mu_i}{\sigma_i}\right)
  \]

This patch is what allows **Entice** to be included at the MOQ floor when `MOQ = 600`.

In [1]:
import math
import itertools
import pandas as pd
import numpy as np

TOTAL_Q1 = 10000
MOQ = 600   # change freely

data = pd.DataFrame([
    ('Gail',      1017,  388),
    ('Isis',      1042,  646),
    ('Entice',    1358,  496),
    ('Assault',   2525,  680),
    ('Teri',      1100,  762),
    ('Electra',   2150,  807),
    ('Stephanie', 1113, 1048),
    ('Seduced',   4017, 1113),
    ('Anita',     3296, 2094),
    ('Daphne',    2383, 1394),
], columns=['style', 'mu', 'sigma'])

data

,style,mu,sigma
0,Gail,1017,388
1,Isis,1042,646
2,Entice,1358,496
3,Assault,2525,680
4,Teri,1100,762
5,Electra,2150,807
6,Stephanie,1113,1048
7,Seduced,4017,1113
8,Anita,3296,2094
9,Daphne,2383,1394


In [2]:
def Phi(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

styles = data['style'].tolist()
mu = dict(zip(data['style'], data['mu']))
sigma = dict(zip(data['style'], data['sigma']))

def evaluate_partition(active_set, bound_set, deferred_set, total_q1=TOTAL_Q1, moq=MOQ):
    active_set = list(active_set)
    bound_set = list(bound_set)
    deferred_set = list(deferred_set)

    if (set(active_set) & set(bound_set)) or (set(active_set) & set(deferred_set)) or (set(bound_set) & set(deferred_set)):
        return None
    if set(active_set) | set(bound_set) | set(deferred_set) != set(styles):
        return None

    q1 = {s: 0.0 for s in styles}
    for s in bound_set:
        q1[s] = float(moq)

    fixed_bound_qty = len(bound_set) * moq
    remaining_qty = total_q1 - fixed_bound_qty
    if remaining_qty < -1e-9:
        return None

    if active_set:
        mu_sum = sum(mu[s] for s in active_set)
        sig_sum = sum(sigma[s] for s in active_set)
        z = (remaining_qty - mu_sum + len(active_set) * moq) / sig_sum
        for s in active_set:
            q1[s] = mu[s] + z * sigma[s] - moq
        if any(q1[s] < moq - 1e-9 for s in active_set):
            return None
        active_risk = Phi(z)
    else:
        if abs(remaining_qty) > 1e-9:
            return None
        z = None
        active_risk = np.nan

    included = active_set + bound_set
    if not included:
        return None

    risks = {}
    for s in active_set:
        risks[s] = active_risk
    for s in bound_set:
        risks[s] = Phi((2*moq - mu[s]) / sigma[s])

    max_risk = max(risks.values())
    risk_sd = float(np.std(list(risks.values()), ddof=0))
    risk_range = max(risks.values()) - min(risks.values())

    q2_expected = {s: mu[s] - q1[s] for s in styles}

    return {
        "active_set": tuple(sorted(active_set)),
        "bound_set": tuple(sorted(bound_set)),
        "deferred_set": tuple(sorted(deferred_set)),
        "z": z,
        "q1": q1,
        "q2_expected": q2_expected,
        "risks": risks,
        "active_risk": active_risk,
        "max_risk": max_risk,
        "risk_sd": risk_sd,
        "risk_range": risk_range,
        "n_included": len(included),
    }

records = []
for mask_active in range(1 << len(styles)):
    active = [styles[i] for i in range(len(styles)) if (mask_active >> i) & 1]
    remaining = [s for s in styles if s not in active]
    for mask_bound in range(1 << len(remaining)):
        bound = [remaining[i] for i in range(len(remaining)) if (mask_bound >> i) & 1]
        deferred = [s for s in remaining if s not in bound]
        rec = evaluate_partition(active, bound, deferred)
        if rec is not None:
            records.append(rec)

search_results = pd.DataFrame([
    {
        "active_set": ", ".join(rec["active_set"]),
        "bound_set": ", ".join(rec["bound_set"]),
        "deferred_set": ", ".join(rec["deferred_set"]),
        "n_included": rec["n_included"],
        "z": rec["z"],
        "active_risk": rec["active_risk"],
        "max_risk": rec["max_risk"],
        "risk_sd": rec["risk_sd"],
        "risk_range": rec["risk_range"],
    }
    for rec in records
]).sort_values(
    ["max_risk", "risk_sd", "risk_range", "deferred_set"]
).reset_index(drop=True)

search_results.head(20)

,active_set,bound_set,deferred_set,n_included,z,active_risk,max_risk,risk_sd,risk_range
0,"Anita, Assault, Daphne, Electra, Seduced",Entice,"Gail, Isis, Stephanie, Teri",6,-0.323752,0.373063,0.375034,7.347211e-04,0.001971
1,"Anita, Assault, Daphne, Electra, Seduced",,"Entice, Gail, Isis, Stephanie, Teri",5,-0.225197,0.410913,0.410913,5.551115e-17,0.000000
2,"Anita, Assault, Daphne, Entice, Seduced",Electra,"Gail, Isis, Stephanie, Teri",6,-0.204085,0.419143,0.419143,1.116489e-01,0.299586
3,"Anita, Assault, Daphne, Seduced","Electra, Entice","Gail, Isis, Stephanie, Teri",6,-0.193335,0.423348,0.423348,1.110255e-01,0.303791
4,"Anita, Assault, Electra, Entice, Seduced",Daphne,"Gail, Isis, Stephanie, Teri",6,-0.182274,0.427684,0.427684,8.558265e-02,0.229642
5,"Anita, Assault, Electra, Seduced","Daphne, Entice","Gail, Isis, Stephanie, Teri",6,-0.167874,0.433341,0.433341,8.602144e-02,0.235300
6,"Anita, Daphne, Electra, Entice, Seduced",Assault,"Gail, Isis, Stephanie, Teri",6,-0.136179,0.445840,0.445840,1.565859e-01,0.420164
7,"Anita, Daphne, Electra, Seduced","Assault, Entice","Gail, Isis, Stephanie, Teri",6,-0.119453,0.452458,0.452458,1.558670e-01,0.426783
8,"Anita, Assault, Daphne, Entice, Seduced",,"Electra, Gail, Isis, Stephanie, Teri",5,-0.100225,0.460083,0.460083,0.000000e+00,0.000000
9,"Anita, Assault, Daphne, Seduced",Entice,"Electra, Gail, Isis, Stephanie, Teri",5,-0.079720,0.468230,0.468230,3.727823e-02,0.093196


In [3]:
best = min(
    records,
    key=lambda rec: (
        rec["max_risk"],
        rec["risk_sd"],
        rec["risk_range"],
        len(rec["deferred_set"]),
        -rec["n_included"]
    )
)

print("Best partition")
print("Active   :", best["active_set"])
print("Bound    :", best["bound_set"])
print("Deferred :", best["deferred_set"])
print("z        :", None if best["z"] is None else round(best["z"], 6))
print("Active risk Phi(z):", None if best["z"] is None else round(best["active_risk"], 6))
print("Max included risk :", round(best["max_risk"], 6))
print("Risk SD           :", round(best["risk_sd"], 6))

Best partition
Active   : ('Anita', 'Assault', 'Daphne', 'Electra', 'Seduced')
Bound    : ('Entice',)
Deferred : ('Gail', 'Isis', 'Stephanie', 'Teri')
z        : -0.323752
Active risk Phi(z): 0.373063
Max included risk : 0.375034
Risk SD           : 0.000735


In [4]:
solution = data.copy()
solution["state"] = solution["style"].apply(
    lambda s: "active" if s in best["active_set"] else ("bound" if s in best["bound_set"] else "deferred")
)
solution["q1"] = solution["style"].map(best["q1"])
solution["q2_expected"] = solution["style"].map(best["q2_expected"])
solution["risk_P(D<q1+MOQ)"] = solution["style"].map(lambda s: best["risks"].get(s, np.nan))
solution = solution[["style", "mu", "sigma", "state", "q1", "q2_expected", "risk_P(D<q1+MOQ)"]]
solution.sort_values(["state", "q1"], ascending=[True, False]).reset_index(drop=True)

,style,mu,sigma,state,q1,q2_expected,risk_P(D<q1+MOQ)
0,Seduced,4017,1113,active,3056.664422,960.335578,0.373063
1,Anita,3296,2094,active,2018.064060,1277.935940,0.373063
2,Assault,2525,680,active,1704.848883,820.151117,0.373063
3,Daphne,2383,1394,active,1331.690210,1051.309790,0.373063
4,Electra,2150,807,active,1288.732424,861.267576,0.373063
5,Entice,1358,496,bound,600.000000,758.000000,0.375034
6,Gail,1017,388,deferred,0.000000,1017.000000,NaN
7,Isis,1042,646,deferred,0.000000,1042.000000,NaN
8,Teri,1100,762,deferred,0.000000,1100.000000,NaN
9,Stephanie,1113,1048,deferred,0.000000,1113.000000,NaN


## What you should see for `MOQ = 600`

The patched search should return:

- **Bound**: Entice
- **Active**: Assault, Electra, Seduced, Anita, Daphne
- **Deferred**: Gail, Isis, Teri, Stephanie

with first-round quantities close to:

- Entice = 600
- Assault = 1704.8
- Electra = 1288.7
- Seduced = 3056.7
- Anita = 2018.1
- Daphne = 1331.7

and included-style risks around **0.373 to 0.375**.

This is the patched behavior you asked for: a near-binding style is allowed to sit at the MOQ floor instead of being dropped.